<a href="https://colab.research.google.com/github/rlagosb/GastricCancerIncidence/blob/main/3_Incidence_estimations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción
Este notebook presenta las estimaciones de incidencia de cáncer gástrico calculadas en el notebook anterior.


# Setup 💾

In [1]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import shapiro, mannwhitneyu, norm, t, skewnorm, kstest

# librerías de python para correr R
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
%load_ext rpy2.ipython

# librerías de R
robjects.r('''
install.packages("mgcv")
library(mgcv)
library(dplyr)
library(boot)
''')

# Carpetas de origen y destino
path_data='https://raw.githubusercontent.com/rlagosb/GastricCancerIncidence/main/Data/'
path_output = '/content/output/'

# Nro de habitantes para calcular tasas
HABS = 100000

(as ‘lib’ is unspecified)







	‘/tmp/Rtmparxu0X/downloaded_packages’



Attaching package: ‘dplyr’



    collapse



    filter, lag



    intersect, setdiff, setequal, union




## Cargar datos

In [2]:
cubo = pd.read_parquet(path_data + 'CUBO_CANCER_DIGESTIVO.parquet')[lambda x: x.Provincia!=122] # filtrar provincia Antártica=122
print(cubo.info())

# Población OMS
# Fuente: https://seer.cancer.gov/stdpopulations/world.who.html

poblacion_estandar = (pd.read_excel(path_data + 'Parametros_estimaciones.xlsx', sheet_name='Poblacion_OMS').drop(columns=['Age']).
                      pivot_table(index=['RangoEdad4080'], values='PoblacionOMS', aggfunc='sum', margins=True).reset_index())
print(poblacion_estandar)

# Escenarios de training y testing
escenarios = pd.read_excel(path_data + 'Parametros_estimaciones.xlsx', sheet_name='Escenarios')

# Periodos
periodos_xl = pd.read_excel(path_data + 'Parametros_estimaciones.xlsx', sheet_name='Periodos RPC', header=1)
def cargar_periodos_rpc(df):
  df = (df.rename(columns={'ID':'Año'}).
        melt(id_vars='Año', value_name='Periodo', var_name='Provincia').
        dropna(subset='Periodo'))
  for col in df.columns: df[col] = df[col].astype(int)
  return df
periodos = cargar_periodos_rpc(periodos_xl)
periodos.pivot(index='Año', columns='Provincia', values='Periodo').astype('Int64').fillna(0)


<class 'pandas.core.frame.DataFrame'>
Index: 24156 entries, 0 to 24947
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Provincia             24156 non-null  int64  
 1   Nombre Provincia      24156 non-null  object 
 2   Nombre Region         24156 non-null  object 
 3   Region                24156 non-null  int64  
 4   Macrorregion          24156 non-null  object 
 5   Año                   24156 non-null  int64  
 6   Sexo                  24156 non-null  object 
 7   RangoEdad10           24156 non-null  object 
 8   RangoEdad4080         24156 non-null  object 
 9   MedianaRangoEdad10    24156 non-null  float64
 10  MedianaRangoEdad4080  24156 non-null  float64
 11  Poblacion             24156 non-null  int64  
 12  PoblacionRural        24156 non-null  int64  
 13  A                     24156 non-null  int64  
 14  B                     24156 non-null  int64  
 15  C                     24

Provincia,21,22,23,71,72,73,74,81,83,141,142,151,1000
Año,,,,,,,,,,,,,
2003,2005,2005,2005,0,0,0,0,0,2004,2004,2004,0,2004
2004,2005,2005,2005,0,0,0,0,0,2004,2004,2004,0,2004
2005,2005,2005,2005,0,0,0,0,0,2004,2004,2004,0,2004
2006,2005,2005,2005,0,0,0,0,2007,2007,2007,2007,0,2007
2007,2008,2008,2008,0,0,0,0,2007,2007,2007,2007,0,2007
2008,2008,2008,2008,0,0,0,0,2007,2007,2007,2007,2010,2007
2009,2008,2008,2008,0,0,0,0,2010,2010,2010,2010,2010,2011
2010,2011,2011,2011,0,0,0,0,2010,2010,2010,2010,2010,2011
2011,2011,2011,2011,0,0,0,0,2010,2010,2010,2010,2010,2011


# Outcomes CG

- Hospitalizaciones
- Defunciones
- Hospitalizaciones + defunciones únicas
- Incidencia

In [4]:
# @title 🔰 Table 2: Outcomes

def cargar_tasas_tabla2_sexo():

  # cargar datos por población
  df = escenarios[lambda x: (x.Escenario.isin(['Chile','Escenario1']))].drop(columns=['Tipo']).copy()
  df['Grupo'] = df['Escenario'].map({'Chile':'Objetivo','Escenario1':'Estudio'})
  df = (df.merge(cubo, on=['Año','Provincia']))

  # Calcular numeradores para tasas estandarizadas a nivel nacional
  numeradores = ['Defunciones','Egresos', 'EgresosDefs']
  numeradoresPob = numeradores + ['Poblacion']

  # Agregar a nivel nacional por sexo, edad y grupo estudio/objetivo
  df = (df.groupby(['Grupo','Sexo','RangoEdad4080','Año'])[numeradoresPob].sum().reset_index().
        merge(poblacion_estandar, on='RangoEdad4080', how='left')) # agregar población estándar
  for num in numeradores:
    df[num+'OMS'] = (df[num]/df.Poblacion) * df.PoblacionOMS

  # Agregar a nivel de población estudio/objetivo y sexo
  df = (df.groupby(['Grupo','Sexo'])[numeradoresPob + [num+'OMS' for num in numeradoresPob]].sum().reset_index())

  # Agregar total por poblacion
  totals = df.groupby(['Grupo'])[numeradoresPob + [num+'OMS' for num in numeradoresPob]].sum().reset_index()
  totals['Sexo'] = 'Total'
  df = pd.concat([df,totals])

  # Calcular tasas crudas y estandarizadas (Std)
  tasas = ['Mortalidad','Hospitalizaciones','HospDefunciones']
  for tasa, numerador in dict(zip(tasas, numeradores)).items():
    df[tasa] = (df[numerador] / df.Poblacion * HABS)
    df[tasa+'Std'] = (df[numerador+'OMS'] / df.PoblacionOMS * HABS)

  # Tasas como filas
  df = (df.melt(id_vars=['Grupo','Sexo'],
             value_vars=tasas + [tasa +'Std' for tasa in tasas],
             var_name='Tasa', value_name='Valor').
        sort_values(by='Sexo').
        pivot(index='Tasa',columns=['Sexo','Grupo'],values='Valor').
        round(1).reset_index())

  # Ordenar tabla
  df['Tipo'] = 'Cruda'
  df.loc[df.Tasa.str.contains('Std'),'Tipo'] = 'Estándar'
  df = df.sort_values(by='Tipo').set_index(['Tipo','Tasa'])

  return df

cargar_tasas_tabla2_sexo()

Sexo                           Hombre             Mujer            Total  \
Grupo                         Estudio Objetivo Objetivo Estudio Objetivo   
Tipo     Tasa                                                              
Cruda    HospDefunciones         37.9     32.6     17.0    18.5     24.7   
         Hospitalizaciones       25.9     21.6     10.9    12.3     16.2   
         Mortalidad              27.9     24.3     11.8    12.7     18.0   
Estándar HospDefuncionesStd      37.9     30.8     12.8    15.2     21.8   
         HospitalizacionesStd    25.3     20.2      8.6    10.4     14.4   
         MortalidadStd           27.8     22.9      8.5     9.9     15.7   

Sexo                                   
Grupo                         Estudio  
Tipo     Tasa                          
Cruda    HospDefunciones         28.1  
         Hospitalizaciones       19.1  
         Mortalidad              20.3  
Estándar HospDefuncionesStd      26.5  
         HospitalizacionesStd    17.8  
         MortalidadStd           18.8

In [5]:
# ⚠️ `Poblacion` equivale a los Años-personas del periodo

def cargar_datos_Chile(escenario, RangoEdad='RangoEdad4080'):

  # Cargar provincias del escenario
  provs = escenarios[escenarios.Escenario=='Chile'].copy()
  # Cargar datos cubo y cruzar con escenario
  data = cubo.merge(provs, on=['Provincia','Año'], how='inner')

  # Agrupar cubo en RangoEdad seleccionado
  metricas = ['Poblacion','Casos', 'Defunciones']
  group_columns = ['Sexo',RangoEdad,'Mediana'+RangoEdad,'Periodo','Provincia','Grupo','Escenario','Tipo']
  agg_functions = {col: 'sum' for col in metricas}
  agg_functions['RPC'] = 'max'
  data = data.groupby(group_columns, dropna=False).agg(agg_functions).reset_index()

  # Filtrar observaciones para entrenamiento y proyección
  print('Catos Chile (proyección)')
  data = data[(data.RPC>0) | (data.Grupo=='Testing')]
  # Llave predicciones
  data['Provincia_year'] = data.Provincia.astype(str) + '_' + data.Periodo.astype(str)
  # Separar conjuntos
  data_training = data[(data.Grupo=='Training')&(data.Casos>0)].copy()
  data_proyeccion = data[lambda x: x.Grupo=='Testing'].copy()

  for group in ['Training','Testing']:
    print(f'{group}: {data[data.Grupo==group].Periodo.min()} - {data[data.Grupo==group].Periodo.max()} {data[data.Grupo==group].Provincia.unique()} N={len(data[data.Grupo==group])}')

  return data_training, data_proyeccion

df_train, df_pred = cargar_datos_Chile('RangoEdad4080')

Catos Chile (proyección)
Training: 2004 - 2018 [ 83 141 142  21  22  23  81 151  71  72  73  74] N=420
Testing: 2004 - 2023 [   11    14    21    22    23    31    32    33    41    42    43    51
    52    53    54    55    56    57    58    61    62    63    71    72
    73    74    81    82    83    91    92   101   102   103   104   111
   112   113   114   121   123   124   132   133   134   135   136   141
   142   151   152   161   162   163 13109 13110 13111 13112 13113 13114] N=5040


In [6]:
%%R
# Función estadística: recibe datos con índices bootstrap y devuelve
# N estimado por provincia
stat_N_provincia <- function(data_train, indices, all_provs, data_predict) {

  # Remuestrear filas (bootstrap no paramétrico sobre observaciones)
  d <- data_train[indices, ]

  # Handle cases where the bootstrap sample might be too small or empty
  if (nrow(d) == 0 || length(unique(d$Provincia_year)) == 0) {
    return(setNames(rep(NA, length(all_provs)), all_provs))
  }

  # Reajustar el modelo con la muestra bootstrap
  fit_b <- gam(
      Defunciones ~ MedianaRangoEdad4080 + s(Provincia, bs = "re"),
      offset = log(Casos),
      family = quasipoisson,
      data   = d,
      method = "REML"
    )

  if (is.null(fit_b)) {return(setNames(rep(NA, length(all_provs)), all_provs))}

  # Recalcular phi_b y su2_b
  phi_b    <- summary(fit_b)$dispersion
  # Ensure fit_b$sp exists and is valid to prevent errors
  if (length(fit_b$sp) == 0 || is.na(fit_b$sp[1]) || fit_b$sp[1] == 0) {
    su2_b <- NA
  } else {
    su2_b <- phi_b / fit_b$sp[1]
  }

  if (is.na(su2_b)) { # If su2_b calculation failed
    return(setNames(rep(NA, length(all_provs)), all_provs))
  }

  Xp_b <- predict(
    fit_b,
    newdata = data_predict,
    type    = "lpmatrix",
    exclude = "s(Provincia)"
  )

  # Check for valid coefficients
  if (is.null(coef(fit_b))) {
    return(setNames(rep(NA, length(all_provs)), all_provs))
  }

  linear_predictor_boot <- as.vector(Xp_b %*% coef(fit_b))

  # Ensure linear_predictor_boot is numeric and finite
  if (!is.numeric(linear_predictor_boot) || any(!is.finite(linear_predictor_boot))) {
    return(setNames(rep(NA, length(all_provs)), all_provs))
  }

  N_b   <- data_predict$Defunciones / exp(linear_predictor_boot + su2_b / 2)

  # Calculate sums for provinces present in the current bootstrap sample
  current_sums <- tapply(N_b, data_predict$Provincia_year, sum)

  # Create a named vector for all original provinces, initialized to NA
  # Then fill in the calculated sums for the provinces that are present
  full_sums <- setNames(rep(NA_real_, length(all_provs)), all_provs)
  full_sums[names(current_sums)] <- current_sums

  # retorna vector con N por provincia, de longitud consistente
  return(full_sums)
}

In [7]:
%%R -i df_train,df_pred -o ic_boot_B

# Define all unique provinces from the predict dataset. This list needs to be consistent.
provs_all_original <- unique(df_pred$Provincia_year)

calculate_confidence_intervals <- function(data_train, data_predict) {

# Ejecutar bootstrap (R = réplicas; aumentar a 999 para publicación)
set.seed(2024)
boot_res <- boot(
  data = df_train,
  data_predict = df_pred,
  statistic = stat_N_provincia,
  R         = 999,
  all_provs = provs_all_original # Pass the full list of provinces as an extra argument
)

# Extraer IC por provincia
ic_boot_B <- do.call(rbind, lapply(seq_along(provs_all_original), function(k) {
  ci <- boot.ci(boot_res, type = "perc", index = k)
  data.frame(
    Provincia_year  = provs_all_original[k],
    N_estimado = boot_res$t0[k],
    IC95_low   = ci$percent[4],
    IC95_high  = ci$percent[5]
  )
}))

  return(ic_boot_B)
}

ic_boot_B <- calculate_confidence_intervals(df_train, df_pred)

In [8]:
def tabla_provincia_periodo(ic_boot_B):

  # Preparar campos para cruce con cubo
  inc = ic_boot_B.copy()
  inc[['Provincia', 'Periodo']] = inc['Provincia_year'].str.split('_', expand=True)
  inc['Provincia'] = inc['Provincia'].astype(int)
  inc['Periodo'] = inc['Periodo'].astype(int)

  region_map = {15:1, 1:2, 2:3, 3:4, 4:5, 5:6, 13:7, 6:8, 7:9,16:10,8:11, 9:12,14:13,10:14, 11:15, 12:16}
  cubo['RegionOrd'] = cubo.Region.map(region_map)

  # Agregar dimensiones
  inc = (inc.merge(cubo.merge(periodos[periodos.Provincia==1000][['Año','Periodo']], on='Año').
                  groupby(['RegionOrd','Nombre Region','Provincia','Nombre Provincia','Periodo'])[['Poblacion','Defunciones']].sum().reset_index(),
                  on=['Periodo','Provincia'], how='right'))

  # Calcular Incidencia
  inc['Incidencia'] = (inc['N_estimado'] / inc['Poblacion'] * HABS).round(1)
  inc['Lower_CI'] = (inc['IC95_low'] / inc['Poblacion'] * HABS).round(1)
  inc['Upper_CI'] = (inc['IC95_high'] / inc['Poblacion'] * HABS).round(1)
  # Estimation (Lower_CI, Upper_CI)
  inc['Incidencia'] = inc['Incidencia'].astype(str) + ' (' + inc['Lower_CI'].astype(str) + '-' + inc['Upper_CI'].astype(str) + ')'

  return inc


inc = tabla_provincia_periodo(ic_boot_B)

In [10]:
# @title ⭐ Table 3: Incidencia Cruda
inc.pivot(index=['RegionOrd','Nombre Region','Provincia','Nombre Provincia'],columns='Periodo', values='Incidencia').round(1)

Periodo                                                                                         2004  \
RegionOrd Nombre Region                             Provincia Nombre Provincia                         
1         Arica y Parinacota                        151       Arica                 18.8 (18.2-19.7)   
                                                    152       Parinacota               0.0 (0.0-0.0)   
2         Tarapacá                                  11        Iquique               15.4 (14.9-16.1)   
                                                    14        Tamarugal                6.8 (6.6-7.1)   
3         Antofagasta                               21        Antofagasta           16.3 (15.8-17.0)   
                                                    22        El Loa                   9.0 (8.7-9.5)   
                                                    23        Tocopilla             23.2 (22.4-24.2)   
4         Atacama                                   31        Copiapó               13.3 (12.8-13.9)   
                                                    32        Chañaral              12.8 (12.3-13.4)   
                                                    33        Huasco                23.6 (22.7-24.5)   
5         Coquimbo                                  41        Elqui                 20.0 (19.3-20.8)   
                                                    42        Choapa                30.8 (29.8-32.2)   
                                                    43        Limarí                23.7 (22.8-24.6)   
6         Valparaíso                                51        Valparaíso            25.8 (24.9-26.9)   
                                                    52        Isla de Pascua           0.0 (0.0-0.0)   
                                                    53        Los Andes             20.3 (19.6-21.1)   
                                                    54        Petorca               31.1 (30.1-32.4)   
                                                    55        Quillota              30.4 (29.4-31.7)   
                                                    56        San Antonio           32.1 (31.1-33.5)   
                                                    57        San Felipe            19.1 (18.4-19.9)   
                                                    58        Marga Marga           27.7 (26.8-28.9)   
7         Metropolitana de Santiago                 132       Cordillera            12.3 (11.9-12.9)   
                                                    133       Chacabuco             16.8 (16.3-17.6)   
                                                    134       Maipo                 20.9 (20.2-21.8)   
                                                    135       Melipilla             19.9 (19.2-20.7)   
                                                    136       Talagante             20.1 (19.5-21.0)   
                                                    13109     Santiago Norte        22.9 (22.1-23.9)   
                                                    13110     Santiago Occidente    20.9 (20.2-21.8)   
                                                    13111     Santiago Central      16.4 (15.8-17.1)   
                                                    13112     Santiago Oriente      16.0 (15.5-16.7)   
                                                    13113     Santiago Sur          25.6 (24.7-26.7)   
                                                    13114     Santiago Sur Oriente  20.7 (20.1-21.7)   
8         Libertador General Bernardo O'Higgins     61        Cachapoal             21.1 (20.4-22.0)   
                                                    62        Cardenal Caro         19.2 (18.4-20.0)   
                                                    63        Colchagua             35.3 (34.1-36.9)   
9         Maule                                     71        Talca                 36.0 (34.8-37.6)   
                                                

## 📊 Material sup 5: Incidencia cruda por provincia y sexo



In [13]:
def cargar_predicciones(**kwargs):
  predicciones = pd.read_parquet(path_data + 'Predicciones.parquet')

  # filtra predicciones por variables que se pasan en kwargs (ejemplo: Metodo=RIMcl)
  for variable, valor in kwargs.items():
    try:
      filtro = (predicciones[variable]==valor)
      predicciones = predicciones[filtro]
    except: print('No se encontró parámetro: '+variable)
  return predicciones

inc_nacional_std = (pd.concat([cargar_predicciones(Escenario='Chile', Grupo='Testing', Metodo='RIMclHombres'),
                               cargar_predicciones(Escenario='Chile', Grupo='Testing', Metodo='RIMclMujeres')]).
                    merge(poblacion_estandar, on='RangoEdad4080'))
inc_nacional_std['Casos'] = (inc_nacional_std.IncidenciaEst / HABS * inc_nacional_std.PoblacionOMS)
inc_nacional_std = inc_nacional_std.groupby('Periodo')[['Casos','PoblacionOMS']].sum().reset_index()
inc_nacional_std['IncidenciaStd'] = inc_nacional_std.Casos / inc_nacional_std.PoblacionOMS * HABS
print(inc_nacional_std[['Periodo','IncidenciaStd']].round(1))
print(f'Reducción total: {(inc_nacional_std.IncidenciaStd.loc[6]/inc_nacional_std.IncidenciaStd.loc[0]*100).round(2)}%')

   Periodo  IncidenciaStd
0     2004           24.5
1     2007           23.5
2     2011           21.6
3     2014           19.0
4     2017           16.5
5     2020           14.8
6     2023           13.6
Reducción total: 55.52%


In [15]:
def cargar_predicciones(**kwargs):
  predicciones = pd.read_parquet(path_data + 'Predicciones.parquet')

  # filtra predicciones por variables que se pasan en kwargs (ejemplo: Metodo=RIMcl)
  for variable, valor in kwargs.items():
    try:
      filtro = (predicciones[variable]==valor)
      predicciones = predicciones[filtro]
    except: print('No se encontró parámetro: '+variable)
  return predicciones

# Predicciones para Chile
inc = (pd.concat([cargar_predicciones(Escenario='Chile', Grupo='Testing', Metodo='RIMclHombres'),
                 cargar_predicciones(Escenario='Chile', Grupo='Testing', Metodo='RIMclMujeres')]).
                groupby(['Provincia','Periodo','Sexo'])[['CasosEst','Poblacion']].sum().reset_index().
       sort_values(by=['Sexo','Provincia','Periodo']))
inc['IncidenciaEst'] = inc.CasosEst / inc.Poblacion * HABS
inc = (inc.merge(cubo[['Provincia','Nombre Provincia', 'Region', 'Nombre Region']].drop_duplicates(), on='Provincia').
         pivot(index=['Region','Nombre Region','Provincia','Nombre Provincia'], columns=['Sexo','Periodo'], values='IncidenciaEst').round(1))

inc

Sexo                                                                            Hombre  \
Periodo                                                                           2004   
Region Nombre Region                             Provincia Nombre Provincia              
1      Tarapacá                                  11        Iquique                20.1   
                                                 14        Tamarugal              12.6   
2      Antofagasta                               21        Antofagasta            21.7   
                                                 22        El Loa                 14.0   
                                                 23        Tocopilla              36.2   
3      Atacama                                   31        Copiapó                20.0   
                                                 32        Chañaral               15.4   
                                                 33        Huasco                 30.3   
4      Coquimbo                                  41        Elqui                  24.8   
                                                 42        Choapa                 45.2   
                                                 43        Limarí                 29.0   
5      Valparaíso                                51        Valparaíso             33.8   
                                                 52        Isla de Pascua          0.0   
                                                 53        Los Andes              26.0   
                                                 54        Petorca                51.7   
                                                 55        Quillota               41.5   
                                                 56        San Antonio            46.1   
                                                 57        San Felipe             21.6   
                                                 58        Marga Marga            35.3   
6      Libertador General Bernardo O'Higgins     61        Cachapoal              27.7   
                                                 62        Cardenal Caro          26.8   
                                                 63        Colchagua              49.4   
7      Maule                                     71        Talca                  47.6   
                                                 72        Cauquenes              46.1   
                                                 73        Curicó                 50.2   
                                                 74        Linares                44.2   
8      Biobío                                    81        Concepción             36.3   
                                                 82        Arauco                 38.5   
                                                 83        Bíobío                 45.0   
9      La Araucanía                              91        Cautín                 38.6   
                                                 92        Malleco                58.8   
10     Los Lagos                                 101       Llanquihue             30.9   
                                                 102       Chiloé                 29.6   
                                                 103       Osorno                 42.4   
                                                 104       Palena                  9.3   
11     Aysén del General Carlos Ibáñez del Campo 111       Coyhaique              35.3   
                                                 112       Aysén                  22.5   
                                                 113       Capitán Prat           17.3   
                                                 114       General Carrera        66.5   
12     Magallanes y de la Antártica Chilena      121       Magallanes             27.6   
                                                 123       Tierra del Fuego       56.8   
                                        